In [1]:
import os
import json
from glob import glob
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

/home/octoopt/workspace/projects/personal/data_enrichment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def init_login():
    load_dotenv()
    login(os.getenv("HUGGINGFACE_TOKEN"))
    print("Login successful")

In [3]:
def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.

    Args:
        image_url (str): URL of the image to download
        save_path (str): Local path where to save the image (including filename)
        timeout (int): Request timeout in seconds

    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        with httpx.Client() as client:
            response = client.get(image_url, timeout=timeout)
            response.raise_for_status()  # Raise an exception for bad status codes

            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # Write the image to file
            with open(save_path, "wb") as file:
                file.write(response.content)

        print(f"✓ Downloaded: {save_path}")
        return True

    except httpx.RequestError as e:
        print(f"✗ Failed to download {image_url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error saving image: {e}")
        return False


def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

## Old days 

In [4]:
DATASET_DIR = DATA_DIR / "raw" / "wikipedia_famous_people" / "famous_people.json"

In [9]:
ds = pd.read_json(str(DATASET_DIR))
ds.head()

,name,url,summary,image_url,categories,timestamp
0,Sacheus !Gonteb,https://en.wikipedia.org/wiki/Sacheus_!Gonteb,Rear Admiral Sacheus Randy !Gonteb is a Namibi...,NaN,"[Articles with short description, Living peopl...",2025-08-02 04:53:49.485904
1,1.Cuz,https://en.wikipedia.org/wiki/1.Cuz,"Abas Abdikarim Bakar, better known as 1.Cuz (b...",NaN,"[1997 births, 21st-century male rappers, Artic...",2025-08-02 04:53:51.707057
2,1da Banton,https://en.wikipedia.org/wiki/1da_Banton,"Godson Ominibie Epelle, professionally known a...",NaN,"[1994 births, 21st-century Nigerian musicians,...",2025-08-02 04:53:54.208175
3,1nonly,https://en.wikipedia.org/wiki/1nonly,"Nathan Scott Fuller (born April 6, 2004), bett...",NaN,"[2004 births, 21st-century American male rappe...",2025-08-02 04:53:56.621336
4,1ucid,https://en.wikipedia.org/wiki/1ucid,"Kwadwo Bedihene, known professionally as 1ucid...",NaN,"[20th-century births, Afrobeats musicians, All...",2025-08-02 04:53:59.252879


In [10]:
ds = ds.drop(columns=["image_url"])
ds.head()

,name,url,summary,categories,timestamp
0,Sacheus !Gonteb,https://en.wikipedia.org/wiki/Sacheus_!Gonteb,Rear Admiral Sacheus Randy !Gonteb is a Namibi...,"[Articles with short description, Living peopl...",2025-08-02 04:53:49.485904
1,1.Cuz,https://en.wikipedia.org/wiki/1.Cuz,"Abas Abdikarim Bakar, better known as 1.Cuz (b...","[1997 births, 21st-century male rappers, Artic...",2025-08-02 04:53:51.707057
2,1da Banton,https://en.wikipedia.org/wiki/1da_Banton,"Godson Ominibie Epelle, professionally known a...","[1994 births, 21st-century Nigerian musicians,...",2025-08-02 04:53:54.208175
3,1nonly,https://en.wikipedia.org/wiki/1nonly,"Nathan Scott Fuller (born April 6, 2004), bett...","[2004 births, 21st-century American male rappe...",2025-08-02 04:53:56.621336
4,1ucid,https://en.wikipedia.org/wiki/1ucid,"Kwadwo Bedihene, known professionally as 1ucid...","[20th-century births, Afrobeats musicians, All...",2025-08-02 04:53:59.252879


In [12]:
hf_ds = Dataset.from_pandas(df=ds)
hf_ds

Dataset({
    features: ['name', 'url', 'summary', 'categories', 'timestamp'],
    num_rows: 1300
})

In [14]:
data_id = "minhleduc/wiki_famous_person_00"

hf_ds.push_to_hub(data_id, commit_message="Initial commit")

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.07s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/minhleduc/wiki_famous_person_00/commit/1fd688c7abaa1f6cbf506a834c2a75cbacca7549', commit_message='Initial commit', commit_description='', oid='1fd688c7abaa1f6cbf506a834c2a75cbacca7549', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/minhleduc/wiki_famous_person_00', endpoint='https://huggingface.co', repo_type='dataset', repo_id='minhleduc/wiki_famous_person_00'), pr_revision=None, pr_num=None)

## 02.10.2025

In [6]:
datafile = str(DATA_DIR / "german" / "german_artist_detail.json")
json_data = read_json(datafile)

In [12]:
count = 0

for data in json_data:
    image_url = data["image_urls"]
    if not image_url:
        continue
    image_url = image_url[0]
    name = data["title"].replace(" ", "_")
    download_image(image_url, f"images/{name}.jpg")

✓ Downloaded: images/Heinrich_Aldegrever.jpg
✓ Downloaded: images/Elisabeth_von_Adlerflycht.jpg
✓ Downloaded: images/Albrecht_Altdorfer.jpg
✓ Downloaded: images/Jean_Arp.jpg
✓ Downloaded: images/Asam_brothers.jpg
✓ Downloaded: images/Cosmas_Damian_Asam.jpg
✓ Downloaded: images/Egid_Quirin_Asam.jpg
✓ Downloaded: images/Isidor_Ascheim.jpg
✓ Downloaded: images/Jim_Avignon.jpg
✓ Downloaded: images/Johannes_Baader.jpg
✓ Downloaded: images/Caroline_Bardua.jpg
✓ Downloaded: images/Johann_Wolfgang_Baumgartner.jpg
✓ Downloaded: images/Barthel_Beham.jpg
✓ Downloaded: images/Hans_Bellmer.jpg
✓ Downloaded: images/Ella_Bergmann-Michel.jpg
✓ Downloaded: images/Joseph_Beuys.jpg
✓ Downloaded: images/Anna_and_Bernhard_Blume.jpg
✓ Downloaded: images/Bärbel_Bohley.jpg
✓ Downloaded: images/Eberhard_Bosslet.jpg
✓ Downloaded: images/Erwin_Bowien.jpg
✓ Downloaded: images/Pola_Brändle.jpg
✓ Downloaded: images/Jörg_Breu_the_Elder.jpg
✓ Downloaded: images/Jörg_Breu_the_Younger.jpg
✓ Downloaded: images/Hans_Burg

In [5]:
# datafile = str(DATA_DIR / "german" / "politicians")
datafile = str(DATA_DIR / "wiki_other_nations")
datapaths = glob(datafile + "/**.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_002.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_003.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer_001.json']

In [6]:
politician_profiles = []

for path in datapaths:
    json_data = read_json(file_path=path)
    politician_profiles += json_data


len(politician_profiles)

6167

In [7]:
politician_profiles[10]

{'title': 'Lauri Ylönen',
 'url': 'https://en.wikipedia.org/wiki/Lauri_Yl%C3%B6nen',
 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/ce/The_Rasmus_-_2025250_125352_2025-09-07_ZDF-Fernsehgarten_-_Sven_-_5DS_R_-_0115_-_5DSR3551_%28cropped%3B_Lauri_Yl%C3%B6nen%29.jpg/250px-The_Rasmus_-_2025250_125352_2025-09-07_ZDF-Fernsehgarten_-_Sven_-_5DS_R_-_0115_-_5DSR3551_%28cropped%3B_Lauri_Yl%C3%B6nen%29.jpg',
 'summary': 'Lauri Johannes Ylönen (born 23 April 1979) is a Finnish singer-songwriter, best known as the co-founder and frontman of the Finnish alternative rock band The Rasmus .',
 'infobox': {'Born': '( 1979-04-23 ) 23 April 1979 (age\xa046) Helsinki , Finland',
  'Occupation': 'Singer/songwriter',
  'Instrument(s)': 'Vocals , guitar , piano',
  'Labels': 'Playground Music Universal Music ( Finland )',
  'Website': 'amanda.fm'}}

In [9]:
write_json(data=politician_profiles, 
file_path=DATA_DIR / "wiki_other_nations" / "other_country_singer.json")

✓ Written JSON to: /home/octoopt/workspace/projects/personal/data_enrichment/data/wiki_other_nations/other_country_singer.json


True

In [11]:
count = 0

save_dir = str(DATA_DIR / "images" / "other_nation_wiki")

for idx in tqdm(range(len(politician_profiles))):
    data = politician_profiles[idx]
    image_url = data["image_url"]
    if not image_url:
        continue
    name = data["title"].replace(" ", "_")
    local_path = f"{save_dir}/{name}.jpg"
    data['local_path'] = local_path
    download_image(image_url, local_path)

  0%|          | 2/6167 [00:00<43:59,  2.34it/s]

✓ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/other_nation_wiki/Music_of_Democratic_Republic_of_the_Congo.jpg


  0%|          | 4/6167 [00:01<28:15,  3.63it/s]

✓ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/other_nation_wiki/Music_of_Cuba.jpg


  0%|          | 5/6167 [00:01<27:27,  3.74it/s]

...

✓ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/other_nation_wiki/Aryana_Sayeed.jpg


100%|█████████▉| 6165/6167 [57:48<00:00,  4.10it/s]

✓ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/other_nation_wiki/Ahmad_Zahir.jpg


100%|██████████| 6167/6167 [57:48<00:00,  1.78it/s]

✓ Downloaded: /home/octoopt/workspace/projects/personal/data_enrichment/data/images/other_nation_wiki/Ahmad_Wali.jpg
